# GCI World 2026 - v13 = your reproduction of the 0.77865 file + two evidence-based next steps
Run cells 1-6 exactly as before (they load everything from cache and rebuild the 0.77865 file; nothing is retrained if the caches exist). Then:
* **Cell 7 (about 5 minutes, no retraining):** re-optimises the member weights with nested CV. The submitted blend uses weights fixed in advance from the OLD members (0.24/0.12/0.32/0.16/0.16); the e1 members have since improved (about +0.001 each), so those weights may no longer be best.
* **Cell 8 (about 1.7 hours on Colab CPU):** trains **extra_trees** versions of the five e1 members (same features, views, rounds, seeds; one parameter added) and tests a three-way blend (old + e1 + e1-extra_trees). Evidence for extra_trees (LightGBM, product keys, held-out): core one-hot +0.00078 (replicated +0.00077 with another seed/partition), all-families one-hot +0.00139. The logs of this notebook show that mixing different variants was worth +0.0007 held-out (new 0.77258 + old 0.77202 -> 0.77330); extra_trees changes the training noise, so it is a natural third variant.
* Decision rule for both: nested mean gain >= +0.0002 and positive on >= 4 of 5 folds. The bootstrap interval is printed for information only: with 171k rows it is about +-0.0006 wide (your v10 NULL control), so it cannot detect gains below ~0.0006 and would reject real small gains.
Outputs: `v13_A_reweighted.csv`, `v13_B_old_e1_e1et.csv`, `v13_C_old_e1et_swap.csv` in `gci_output_final/` (written only when a rule passes, always printed).

# GCI World 2026 - FINAL reproduction notebook: rebuilds the submitted file `v9_E1_multiview_swap_old_new.csv` (public AUC 0.77865)
Final = 0.5 x OLD blend + 0.5 x NEW blend, each blend = 0.24 qf_shallow + 0.12 qf_reg + 0.32 v1_allfam_oh + 0.16 v2_allfam_oh + 0.16 v2_mono (rank-transformed predictions).
  OLD members: LightGBM on the original features + nested target encodings of q and the decoded product Q0 (annuity = Q0*credit + F, F ~ 129.5, label-free decode).
  NEW members: the same plus (a) 16 learned-expectation residual models (label-free, fitted on train+test with out-of-fold predictions) and (b) multi-view stacking features
  (small models on EXT / loan / person views, out-of-fold for training rows).
Uses ONLY train.csv, test.csv, sample_submission.csv. No external data, no test labels. Runtime about 3 hours on a Colab CPU runtime; every member is cached to Drive (folder gci_cache_final), so after a disconnect re-run cells 1-2 and the later cells.
Read README_HONORS_FINAL.md for the AI-assistance disclosure and sources.

In [ ]:
# cell 1 - setup
import os, sys, glob, time, warnings; warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount("/content/drive")
except Exception as e: print("Drive not mounted (ok if files are in /content):", e)
import numpy as np, pandas as pd, lightgbm as lgb
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
RUN_E1, RUN_E2, RUN_E3 = True, False, False   # only the final pipeline's parts are used here
SMOKE = os.environ.get("SMOKE") == "1"                   # SMOKE=1 -> tiny quick test of the code path only
def find(name, roots=("/content", "/content/drive/MyDrive", os.getcwd())):
    for r in roots:
        for pat in (name, "*/"+name, "*/*/"+name, "*/*/*/"+name, "*/*/*/*/"+name):
            hit = glob.glob(os.path.join(r, pat))
            if hit: return hit[0]
    return None
def find_data():   # validated finder: 34 columns, TARGET present, 171,202 rows (skips any other train.csv with the same name)
    cands = []
    for r in ("/content", "/content/drive/MyDrive", os.getcwd()):
        for pat in ("train.csv", "*/train.csv", "*/*/train.csv", "*/*/*/train.csv", "*/*/*/*/train.csv"): cands += glob.glob(os.path.join(r, pat))
    for f in sorted(set(cands)):
        d = os.path.dirname(f)
        if not all(os.path.exists(os.path.join(d, x)) for x in ("test.csv", "sample_submission.csv")): continue
        try: head = open(f, "rb").readline().decode("utf-8", "ignore").strip().split(","); nrows = sum(1 for _ in open(f, "rb")) - 1
        except Exception: continue
        if len(head) == 34 and "TARGET" in head and "AMT_ANNUITY" in head and nrows == 171202: print("using competition files in", d); return [os.path.join(d, x) for x in ("train.csv", "test.csv", "sample_submission.csv")]
        print("skipping a different train.csv:", f, "| columns", len(head), "| rows", nrows)
    raise SystemExit("Real competition files not found: upload train.csv, test.csv, sample_submission.csv from the competition input folder to /content")
TRAIN_PATH, TEST_PATH, SAMPLE_PATH = find_data()
assert TRAIN_PATH and TEST_PATH and SAMPLE_PATH, "Could not find train.csv / test.csv / sample_submission.csv - upload them to /content or put them in Drive"
HGB_PATH = find("GCI_hgb_product_test.npy"); HGB_OOF = find("GCI_hgb_product_oof.npy")
base_dir = "/content/drive/MyDrive" if os.path.isdir("/content/drive/MyDrive") else "/content"
CACHE_DIR = os.path.join(base_dir, "gci_cache_final") if not SMOKE else "/tmp/gci_cache_smoke"; OUT_DIR = os.path.join(base_dir, "gci_output_final") if not SMOKE else "/tmp/gci_out_smoke"
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
THREADS = os.cpu_count() or 1
print("train:", TRAIN_PATH, "\ntest:", TEST_PATH, "\nsample:", SAMPLE_PATH, "\nhgb oof:", HGB_OOF, "\nhgb npy:", HGB_PATH, "\ncache:", CACHE_DIR, "| threads:", THREADS)

In [ ]:
# cell 2 - features, q and decoded-product keys
CATS = ["NAME_CONTRACT_TYPE","CODE_GENDER","NAME_INCOME_TYPE","NAME_EDUCATION_TYPE","NAME_FAMILY_STATUS","NAME_HOUSING_TYPE","OCCUPATION_TYPE","ORGANIZATION_TYPE"]
tr = pd.read_csv(TRAIN_PATH); te = pd.read_csv(TEST_PATH); sample = pd.read_csv(SAMPLE_PATH)
y = tr["TARGET"].values; ntr = len(tr)
al = pd.concat([tr.drop(columns=["TARGET"]), te], ignore_index=True)
def new_cols(d): return pd.DataFrame(d, index=al.index)
def base(al):
    X = al.drop(columns=["SK_ID_CURR"]).copy()
    for c in CATS: X[c] = X[c].astype("category").cat.codes.replace(-1, np.nan).astype(float)
    return X
def f_clean(al, X):
    d = {"DE_SENT": (al.DAYS_EMPLOYED == 365243).astype(int), "DAYS_EMPLOYED_CLEAN": al.DAYS_EMPLOYED.where(al.DAYS_EMPLOYED != 365243),
         "ORG_XNA": (al.ORGANIZATION_TYPE == "XNA").astype(int), "CAR_AGE_OUT": (al.OWN_CAR_AGE >= 60).astype(int), "MISS_CNT": al.isna().sum(axis=1)}
    for c in ["EXT_SOURCE_1","EXT_SOURCE_3","OWN_CAR_AGE","AMT_REQ_CREDIT_BUREAU_YEAR"]: d[c + "_NA"] = al[c].isna().astype(int)
    return pd.concat([X, new_cols(d)], axis=1)
def f_ext(al, X):
    e = al[["EXT_SOURCE_1","EXT_SOURCE_2","EXT_SOURCE_3"]]; d = {}
    d["EXT_MEAN"] = e.mean(axis=1); d["EXT_MIN"] = e.min(axis=1); d["EXT_MAX"] = e.max(axis=1); d["EXT_STD"] = e.std(axis=1)
    d["EXT_RANGE"] = d["EXT_MAX"] - d["EXT_MIN"]; d["EXT_NOBS"] = e.notna().sum(axis=1); d["EXT_PROD"] = e.prod(axis=1)
    d["EXT_W"] = (al.EXT_SOURCE_1*2 + al.EXT_SOURCE_2*3 + al.EXT_SOURCE_3*4) / 9
    d["E12"] = al.EXT_SOURCE_1*al.EXT_SOURCE_2; d["E13"] = al.EXT_SOURCE_1*al.EXT_SOURCE_3; d["E23"] = al.EXT_SOURCE_2*al.EXT_SOURCE_3
    d["D12"] = al.EXT_SOURCE_1-al.EXT_SOURCE_2; d["D13"] = al.EXT_SOURCE_1-al.EXT_SOURCE_3; d["D23"] = al.EXT_SOURCE_2-al.EXT_SOURCE_3
    return pd.concat([X, new_cols(d)], axis=1)
def f_fin(al, X):
    inc, cr, an, gp = al.AMT_INCOME_TOTAL, al.AMT_CREDIT, al.AMT_ANNUITY, al.AMT_GOODS_PRICE
    d = {"CREDIT_INC": cr/inc, "ANN_INC": an/inc, "ANN_CREDIT": an/cr, "TERM": cr/an, "GOODS_CREDIT": gp/cr, "CREDIT_MINUS_GOODS": cr-gp, "GOODS_INC": gp/inc,
         "INC_PER_FAM": inc/al.CNT_FAM_MEMBERS, "CREDIT_PER_FAM": cr/al.CNT_FAM_MEMBERS, "ANN_PER_FAM": an/al.CNT_FAM_MEMBERS}
    for c in ["AMT_INCOME_TOTAL","AMT_CREDIT","AMT_ANNUITY","AMT_GOODS_PRICE"]: d["LOG_" + c] = np.log1p(al[c])
    return pd.concat([X, new_cols(d)], axis=1)
def f_age(al, X):
    age = -al.DAYS_BIRTH; de = al.DAYS_EMPLOYED.where(al.DAYS_EMPLOYED != 365243)
    d = {"AGE": age/365.25, "EMP_AGE": de/al.DAYS_BIRTH, "EMP_YEARS": -de/365.25, "REG_AGE": al.DAYS_REGISTRATION/al.DAYS_BIRTH, "ID_AGE": al.DAYS_ID_PUBLISH/al.DAYS_BIRTH,
         "PHONE_AGE": al.DAYS_LAST_PHONE_CHANGE/al.DAYS_BIRTH, "CAR_AGE_RATIO": al.OWN_CAR_AGE/(age/365.25), "CREDIT_PER_AGE": al.AMT_CREDIT/(age/365.25), "INC_PER_AGE": al.AMT_INCOME_TOTAL/(age/365.25)}
    return pd.concat([X, new_cols(d)], axis=1)
def f_bureau(al, X):
    b = [c for c in al.columns if c.startswith("AMT_REQ")]
    d = {"BUR_SUM": al[b].sum(axis=1, min_count=1), "BUR_RECENT": al[["AMT_REQ_CREDIT_BUREAU_HOUR","AMT_REQ_CREDIT_BUREAU_MON"]].sum(axis=1, min_count=1),
         "BUR_MON_YEAR": al.AMT_REQ_CREDIT_BUREAU_MON/(al.AMT_REQ_CREDIT_BUREAU_YEAR+1)}
    return pd.concat([X, new_cols(d)], axis=1)
def f_count(al, X):
    d = {"CNT_" + c: al[c].map(al[c].value_counts()) for c in CATS + ["REGION_POPULATION_RELATIVE","DAYS_BIRTH","AMT_CREDIT","AMT_GOODS_PRICE","AMT_ANNUITY","AMT_INCOME_TOTAL"]}
    return pd.concat([X, new_cols(d)], axis=1)
def f_group(al, X):
    tmp = al.copy(); tmp["CREDIT_INC"] = al.AMT_CREDIT/al.AMT_INCOME_TOTAL; d = {}
    for k in ["ORGANIZATION_TYPE","OCCUPATION_TYPE","NAME_INCOME_TYPE","NAME_EDUCATION_TYPE","REGION_POPULATION_RELATIVE"]:
        for v in ["AMT_INCOME_TOTAL","AMT_CREDIT","EXT_SOURCE_1","EXT_SOURCE_2","EXT_SOURCE_3","DAYS_BIRTH","CREDIT_INC"]:
            d[f"G_{k[:6]}_{v[-6:]}_DIFF"] = tmp[v] - tmp.groupby(k)[v].transform("mean")
    return pd.concat([X, new_cols(d)], axis=1)
def f_kdec(al, X):    # credit = goods * (1 + 0.0132*k) for ~90% of cash loans with credit > goods -> integer k (insurance/tariff tier); NaN where another tariff family applies
    C, G = al.AMT_CREDIT, al.AMT_GOODS_PRICE; cash = al.NAME_CONTRACT_TYPE == "Cash loans"; kf = (C / G - 1) / 0.0132; k = kf.round(); ok = cash & (C > G) & ((kf - k).abs() < 0.1)
    K = pd.Series(np.where(ok, k, np.where(cash & (C == G), 0.0, np.nan)), index=al.index)
    return pd.concat([X, new_cols({"K_DEC": K, "CG_RATIO": C / G})], axis=1)
FAMS = {"kdec": f_kdec, "clean": f_clean, "ext": f_ext, "fin": f_fin, "age": f_age, "bureau": f_bureau, "count": f_count, "group": f_group}
def build(fams):
    X = base(al)
    for f in fams: X = FAMS[f](al, X)
    X = X.replace([np.inf, -np.inf], np.nan)
    return X.iloc[:ntr].reset_index(drop=True), X.iloc[ntr:].reset_index(drop=True)
CORE = ["clean", "fin", "age", "bureau"]; ALLF = ["clean", "ext", "fin", "age", "bureau", "count", "group"]
Xc, Xct = build(CORE); Xa, Xat = build(ALLF)                              # members WITHOUT k
Xck, Xckt = build(CORE + ["kdec"]); Xak, Xakt = build(ALLF + ["kdec"])   # members WITH decoded k (v2 members)
# keep the frames WITHOUT residual features; residuals are attached per experiment
BASEF = dict(Xc=Xc, Xct=Xct, Xa=Xa, Xat=Xat, Xck=Xck, Xckt=Xckt, Xak=Xak, Xakt=Xakt)
if SMOKE: y = y[:8000]
n, nt = len(y), len(Xct); tidx = np.arange(ntr, ntr + nt) if not SMOKE else np.arange(8000, 8000 + nt)
# ---- q, and the decoded product: annuity = Q0*credit + F  (F ~ 129.5 monthly fee on ~75% of cash loans); label-free ----
q = al.AMT_ANNUITY / al.AMT_CREDIT; _cash = (al.NAME_CONTRACT_TYPE == "Cash loans").values; _qb = (al.AMT_ANNUITY - 129.5) / al.AMT_CREDIT
_q5, _qb5 = q.round(5), _qb.round(5)
FEE = ((_qb5.map(_qb5.value_counts()).values > _q5.map(_q5.value_counts()).values) & _cash).astype(int); Q0 = pd.Series(np.where(FEE, _qb, q))
KS = {"Q3": q.round(3).astype(str), "Q4": q.round(4).astype(str), "Q5": q.round(5).astype(str),
      "P3": Q0.round(3).astype(str), "P4": Q0.round(4).astype(str), "P5": Q0.round(5).astype(str), "PF4": Q0.round(4).astype(str) + "|" + pd.Series(FEE).astype(str)}
M0 = {"Q3": 100., "Q4": 30., "Q5": 30., "P3": 100., "P4": 30., "P5": 30., "PF4": 30.}
if SMOKE: KS = {k: v.iloc[list(range(8000)) + list(range(ntr, ntr + nt))].reset_index(drop=True) for k, v in KS.items()}
print("decoded products:", Q0.round(5).nunique(), "(q5 distinct:", q.round(5).nunique(), ") | fee share of cash loans:", round(float(FEE[_cash].mean()), 3), "| features core/all:", Xc.shape[1], Xa.shape[1])
def enc(fit, qi, keys=None):
    pr = float(y[fit].mean()); out = {}
    for k in (keys or KS):
        key = KS[k]; g = pd.DataFrame({"g": key.iloc[fit].values, "y": y[fit]}).groupby("g")["y"].agg(["sum", "count"]); s = key.iloc[qi]
        cnt = s.map(g["count"]).fillna(0).to_numpy(float); sm = s.map(g["sum"]).fillna(0).to_numpy(float)
        out[f"TE_{k}"] = (sm + M0[k] * pr) / (cnt + M0[k]); out[f"CNT_{k}"] = cnt
    return pd.DataFrame(out)
def train_enc(a):     # nested out-of-fold encodings for training rows (a row's own label never enters its own feature)
    cols = enc(a[:10], a[:5]).columns; inner = np.zeros((len(a), len(cols)))
    for ia, ib in StratifiedKFold(5, shuffle=True, random_state=7).split(a, y[a]): inner[ib] = enc(a[ia], a[ib]).values
    return pd.DataFrame(inner, columns=cols)
def onehot(Xa_, Xt_):
    z = pd.concat([Xa_, Xt_], ignore_index=True); oh = pd.get_dummies(z[CATS].astype("category"), dummy_na=True, dtype=np.float32)
    oh.columns = [f"OH_{i}" for i in range(oh.shape[1])]; Z = pd.concat([z.drop(columns=CATS), oh], axis=1)
    return Z.iloc[:len(Xa_)].reset_index(drop=True), Z.iloc[len(Xa_):].reset_index(drop=True)
Xc, Xa = BASEF["Xc"], BASEF["Xa"]   # for the print above
BASE = dict(objective="binary", bagging_freq=1, verbose=-1, num_threads=THREADS, seed=42)
rk = lambda v: (rankdata(v) - 1) / (len(v) - 1)
FOLDS = 2 if SMOKE else 5
W = {"qf_shallow": 0.24, "qf_reg": 0.12, "v1_allfam_oh": 0.32, "v2_allfam_oh": 0.16, "v2_mono": 0.16}   # the fixed weights of the submitted blend

# ---- learned-expectation (residual) features; seeds = how many different fold splits are averaged (E2 uses 3) ----
from sklearn.ensemble import HistGradientBoostingRegressor as HR, HistGradientBoostingClassifier as HC
from sklearn.model_selection import KFold, train_test_split
def compute_resid(seeds=(0,)):
    raw = al.drop(columns=["SK_ID_CURR"]).copy()
    for c in CATS: raw[c] = raw[c].astype("category").cat.codes.replace(-1, np.nan).astype(float)
    raw["DAYS_EMPLOYED"] = raw.DAYS_EMPLOYED.where(raw.DAYS_EMPLOYED != 365243)
    demo = [c for c in raw.columns if c.startswith("EXT_SOURCE") or c.startswith("AMT_")]
    T = {"EXT1": (raw.EXT_SOURCE_1, ["EXT_SOURCE_1"]), "EXT2": (raw.EXT_SOURCE_2, ["EXT_SOURCE_2"]), "EXT3": (raw.EXT_SOURCE_3, ["EXT_SOURCE_3"]),
         "LOGINC": (np.log(raw.AMT_INCOME_TOTAL), ["AMT_INCOME_TOTAL"]), "LOGCR": (np.log(raw.AMT_CREDIT), ["AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]),
         "EMP": (raw.DAYS_EMPLOYED, ["DAYS_EMPLOYED"]), "REG": (raw.DAYS_REGISTRATION, ["DAYS_REGISTRATION"]), "AGE": (raw.DAYS_BIRTH, ["DAYS_BIRTH"]),
         "IDP": (raw.DAYS_ID_PUBLISH, ["DAYS_ID_PUBLISH"]), "PHONE": (raw.DAYS_LAST_PHONE_CHANGE, ["DAYS_LAST_PHONE_CHANGE"]), "REGPOP": (raw.REGION_POPULATION_RELATIVE, ["REGION_POPULATION_RELATIVE"]),
         "FAM": (raw.CNT_FAM_MEMBERS, ["CNT_FAM_MEMBERS"]), "CAR": (raw.OWN_CAR_AGE, ["OWN_CAR_AGE"]),
         "EXT1D": (raw.EXT_SOURCE_1, demo), "EXT2D": (raw.EXT_SOURCE_2, demo), "EXT3D": (raw.EXT_SOURCE_3, demo)}
    if SMOKE: T = dict(list(T.items())[:2])
    out = {}; t0 = time.time()
    for nm, (yt, drop_cols) in T.items():
        Xp = raw.drop(columns=drop_cols); pred = np.zeros(len(raw)); obs = yt.notna().values
        for sd in seeds:
            for tri, tei in KFold(2 if SMOKE else 5, shuffle=True, random_state=sd).split(Xp):
                m = HR(max_iter=5 if SMOKE else 120, learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=100, random_state=0).fit(Xp.iloc[tri][obs[tri]], yt.iloc[tri][obs[tri]]); pred[tei] += m.predict(Xp.iloc[tei]) / len(seeds)
        out[f"RP_{nm}"] = pred; out[f"RR_{nm}"] = np.where(obs, yt.values - pred, np.nan); print(f"  residual target {nm} ({time.time()-t0:.0f}s)", flush=True)
    return pd.DataFrame(out)
def get_resid(seeds, name):
    path = os.path.join(CACHE_DIR, name)
    old = os.path.join(base_dir, "gci_cache_v6", "resid16.pkl")
    if not SMOKE and os.path.exists(path): return pd.read_pickle(path)
    if not SMOKE and seeds == (0,) and os.path.exists(old): return pd.read_pickle(old)       # reuse the v6 cache (identical recipe)
    R = compute_resid(seeds)
    if not SMOKE: R.to_pickle(path)
    return R
def with_resid(R):
    R_tr, R_te = R.iloc[:ntr].reset_index(drop=True), R.iloc[ntr:].reset_index(drop=True)
    assert len(R) == ntr + len(BASEF["Xct"])
    return {k: pd.concat([BASEF[k], R_te if k.endswith("t") else R_tr], axis=1) for k in BASEF}
def build_members(f):
    Xc_, Xct_, Xa_, Xat_, Xck_, Xckt_, Xak_, Xakt_ = [f[k] for k in ("Xc", "Xct", "Xa", "Xat", "Xck", "Xckt", "Xak", "Xakt")]
    if SMOKE: Xc_, Xa_, Xck_, Xak_ = [d.iloc[:8000] for d in (Xc_, Xa_, Xck_, Xak_)]
    Xco, Xcot = onehot(Xc_, Xct_); Xao, Xaot = onehot(Xa_, Xat_); Xaok, Xaokt = onehot(Xak_, Xakt_)
    MONO_K = [-1 if c in ("EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3") else 0 for c in Xck_.columns] + [0] * (2 * len(KS))
    return {
     "qf_shallow":   (Xco,  Xcot,  dict(learning_rate=0.05, num_leaves=7, max_depth=3, min_child_samples=150, feature_fraction=0.5, bagging_fraction=0.8, lambda_l2=20), 1300, 1430, [42, 7, 2024, 11, 99]),
     "qf_reg":       (Xco,  Xcot,  dict(learning_rate=0.03, num_leaves=11, min_child_samples=300, feature_fraction=0.4, bagging_fraction=0.8, lambda_l2=40), 1939, 2133, [42, 7, 2024, 11, 99]),
     "v1_allfam_oh": (Xao,  Xaot,  dict(learning_rate=0.04, num_leaves=15, min_child_samples=150, feature_fraction=0.3, bagging_fraction=0.8, lambda_l2=20), 881, 969, [42, 7, 2024]),
     "v2_allfam_oh": (Xaok, Xaokt, dict(learning_rate=0.04, num_leaves=15, min_child_samples=150, feature_fraction=0.3, bagging_fraction=0.8, lambda_l2=20), 881, 969, [42, 7, 2024]),
     "v2_mono":      (Xck_.reset_index(drop=True), Xckt_.reset_index(drop=True), dict(learning_rate=0.02, num_leaves=15, min_child_samples=150, feature_fraction=0.5, bagging_fraction=0.8, lambda_l2=20, monotone_constraints=MONO_K), 1571, 1729, [42, 7, 2024])}
def run_member(name, spec, tag, view=None):
    X, Xt, p, R, Rf, seeds = spec[name]; f = os.path.join(CACHE_DIR, f"{tag}_{name}.npz")
    if view is not None and "monotone_constraints" in p: p = {**p, "monotone_constraints": list(p["monotone_constraints"]) + [0] * view["tr"][0].shape[1]}   # extra view columns are unconstrained
    if os.path.exists(f) and not SMOKE:
        z = np.load(f); print(f"{tag}_{name}: loaded from cache | held-out AUC {roc_auc_score(y, z['oof']):.5f}"); return z["test"], z["oof"]
    tf = np.zeros(nt); oof = np.zeros(n); t0 = time.time(); ex = lambda key, k=None: ([] if view is None else [view[key] if k is None else view[key][k]])
    for k, (a, b) in enumerate(StratifiedKFold(FOLDS, shuffle=True, random_state=42).split(np.zeros(n), y)):
        fa = pd.concat([X.iloc[a].reset_index(drop=True), train_enc(a)] + ex("tr", k), axis=1); fb = pd.concat([X.iloc[b].reset_index(drop=True), enc(a, b)] + ex("va", k), axis=1); ft = pd.concat([Xt.reset_index(drop=True), enc(a, tidx)] + ex("te", k), axis=1)
        assert "monotone_constraints" not in p or len(p["monotone_constraints"]) == fa.shape[1], f"monotone constraints ({len(p['monotone_constraints'])}) != features ({fa.shape[1]})"
        m = lgb.train({**BASE, **p}, lgb.Dataset(fa, y[a]), 60 if SMOKE else R); oof[b] = m.predict(fb); tf += m.predict(ft) / FOLDS
        print(f"  {tag}_{name} fold {k+1}/{FOLDS}: AUC {roc_auc_score(y[b], oof[b]):.5f} ({time.time()-t0:.0f}s)", flush=True)
    allr = np.arange(n); fa = pd.concat([X.reset_index(drop=True), train_enc(allr)] + ex("all_tr"), axis=1); ft = pd.concat([Xt.reset_index(drop=True), enc(allr, tidx)] + ex("all_te"), axis=1); fr = []
    for sd in ([42] if SMOKE else seeds):
        fr.append(rk(lgb.train({**BASE, **p, "seed": sd}, lgb.Dataset(fa, y), 60 if SMOKE else Rf).predict(ft))); print(f"  {tag}_{name} full-data seed {sd} done ({time.time()-t0:.0f}s)", flush=True)
    test = rk(0.5 * rk(tf) + 0.5 * np.mean(fr, 0)); print(f"{tag}_{name}: held-out AUC {roc_auc_score(y, oof):.5f}", flush=True)
    if not SMOKE: np.savez(f, test=test, oof=oof)
    return test, oof
RESV = {}

In [ ]:
# cell 3 - residual features (v6 recipe, one fold seed) and the baseline member frames
RESID1 = get_resid((0,), "resid16.pkl"); F1 = with_resid(RESID1); SPEC1 = build_members(F1)
print("v6-recipe features per member family (core / all-families):", F1["Xc"].shape[1], F1["Xa"].shape[1])

In [ ]:
# cell 4 - E1: multi-view stacking (about 10 min for the view models + about 1.7 h for the five members; switch off in cell 1 if not wanted)
def lg(p): p = np.clip(p, 1e-4, 1 - 1e-4); return np.log(p / (1 - p))
def vf(d):
    df = pd.DataFrame({f"VIEW_{k}": v for k, v in d.items()}); df["VD_EL"] = df.VIEW_EXT - df.VIEW_LOAN; df["VD_EP"] = df.VIEW_EXT - df.VIEW_PERSON; df["VD_LP"] = df.VIEW_LOAN - df.VIEW_PERSON; return df
def compute_views():
    path = os.path.join(CACHE_DIR, "e1_views.pkl")
    if os.path.exists(path) and not SMOKE: return pd.read_pickle(path)
    Xs = F1["Xa"].iloc[:n].copy(); Xts = F1["Xat"].copy(); Xs["Q0num"] = Q0.values[:n]; Xts["Q0num"] = Q0.values[ntr:]
    cols = list(Xs.columns)
    ext = [c for c in cols if "EXT" in c or c in ("E12", "E13", "E23", "D12", "D13", "D23")]
    loan = [c for c in cols if c.startswith(("AMT_", "LOG_", "CREDIT_", "ANN_", "GOODS_", "TERM", "Q0num")) or c in ("CREDIT_MINUS_GOODS", "RP_LOGINC", "RR_LOGINC", "RP_LOGCR", "RR_LOGCR")]
    person = [c for c in cols if c not in set(ext) | set(loan) and not c.startswith(("G_", "CNT_"))]
    views = {"EXT": ext, "LOAN": loan, "PERSON": person}; print("view sizes:", {k: len(v) for k, v in views.items()})
    mk = lambda: HC(max_iter=5 if SMOKE else 100, learning_rate=0.08, max_leaf_nodes=15, min_samples_leaf=150, l2_regularization=20, random_state=0)
    out = {"tr": [], "va": [], "te": []}; t0 = time.time()
    for k, (a, b) in enumerate(StratifiedKFold(FOLDS, shuffle=True, random_state=42).split(np.zeros(n), y)):
        trv, vav, tev = {}, {}, {}
        for vn, vc in views.items():
            inner = np.zeros(len(a))
            for ia, ib in StratifiedKFold(3, shuffle=True, random_state=11).split(a, y[a]): inner[ib] = mk().fit(Xs.iloc[a[ia]][vc], y[a[ia]]).predict_proba(Xs.iloc[a[ib]][vc])[:, 1]
            m = mk().fit(Xs.iloc[a][vc], y[a]); trv[vn] = lg(inner); vav[vn] = lg(m.predict_proba(Xs.iloc[b][vc])[:, 1]); tev[vn] = lg(m.predict_proba(Xts[vc])[:, 1])
        out["tr"].append(vf(trv)); out["va"].append(vf(vav)); out["te"].append(vf(tev)); print(f"  views fold {k+1}/{FOLDS} ({time.time()-t0:.0f}s)", flush=True)
    trv, tev = {}, {}
    for vn, vc in views.items():
        inner = np.zeros(n)
        for ia, ib in StratifiedKFold(3, shuffle=True, random_state=11).split(np.zeros(n), y): inner[ib] = mk().fit(Xs.iloc[ia][vc], y[ia]).predict_proba(Xs.iloc[ib][vc])[:, 1]
        trv[vn] = lg(inner); tev[vn] = lg(mk().fit(Xs[vc], y).predict_proba(Xts[vc])[:, 1])
    out["all_tr"] = vf(trv); out["all_te"] = vf(tev)
    if not SMOKE: pd.to_pickle(out, path)
    return out
if RUN_E1:
    VIEW1 = compute_views()
    for nm in W: RESV[("e1", nm)] = run_member(nm, SPEC1, "e1", VIEW1)

In [ ]:
# cell 5 - OLD members (original features + product keys; no residual / view features)
SPEC0 = build_members(BASEF)
for nm in W: RESV[("old", nm)] = run_member(nm, SPEC0, "old")

In [ ]:
# cell 6 - final blend, held-out check and the submission file
ids = sample.SK_ID_CURR
blend = lambda tag, i: sum(W[k] * (RESV[(tag, k)][0] if i == 1 else rk(RESV[(tag, k)][1])) for k in W)
final = 0.5 * blend("old", 1) + 0.5 * blend("e1", 1)
oof_final = 0.5 * blend("old", 0) + 0.5 * blend("e1", 0)
print(f"held-out AUC of the final blend (weights fixed in advance): {roc_auc_score(y, oof_final):.5f} | old blend {roc_auc_score(y, blend('old', 0)):.5f} | new blend {roc_auc_score(y, blend('e1', 0)):.5f}")
out = pd.DataFrame({"SK_ID_CURR": ids, "TARGET": final}); assert list(out.columns) == ["SK_ID_CURR", "TARGET"] and len(out) == 61500 and out.SK_ID_CURR.is_unique and np.isfinite(out.TARGET).all() and np.array_equal(out.SK_ID_CURR.values, ids.values)
if not SMOKE:
    p = os.path.join(OUT_DIR, "GCI_submission_reproduced_final.csv"); out.to_csv(p, index=False); print("wrote", p)
    orig = find("v9_E1_multiview_swap_old_new*.csv")      # optional: the file you actually submitted (public 0.77865)
    if orig:
        from scipy.stats import spearmanr
        o = pd.read_csv(orig); assert np.array_equal(o.SK_ID_CURR.values, ids.values)
        print("compared with the submitted file:", orig, "| Spearman:", round(spearmanr(o.TARGET, out.TARGET).statistic, 6), "| max |difference|:", float(np.abs(o.TARGET.values - out.TARGET.values).max()))
    else: print("Upload the submitted v9_E1_multiview_swap_old_new.csv to /content to compare automatically.")
    try:
        from google.colab import files as cf; cf.download(p)
    except Exception: print("download manually from", OUT_DIR)
else: print("SMOKE run: file not written")

In [ ]:
# cell 7 - re-optimise the member weights with nested CV (uses cached predictions, no retraining)
from sklearn.model_selection import StratifiedKFold as SKF
COMPS = [(t, nm) for t in ("old", "e1") for nm in W]
O = {c: rk(RESV[c][1]) for c in COMPS}; T = {c: RESV[c][0] for c in COMPS}
fixed_oof = 0.5 * blend("old", 0) + 0.5 * blend("e1", 0); fixed_test = 0.5 * blend("old", 1) + 0.5 * blend("e1", 1)      # = the submitted recipe
def fast_auc(s, rows):
    yy = y[rows]; r = rankdata(s[rows]); npos = yy.sum(); return (r[yy == 1].sum() - npos * (npos + 1) / 2) / (npos * (len(yy) - npos))
def climb(rows, iters=40):          # greedy hill-climb with replacement over the 10 (old/e1 x member) predictions
    cnt = {c: 0 for c in COMPS}; cur = np.zeros(n)
    for _ in range(iters):
        best = max(COMPS, key=lambda c: fast_auc(cur + O[c], rows)); cnt[best] += 1; cur = cur + O[best]
    return {c: v / iters for c, v in cnt.items() if v}
apply_w = lambda w, D: sum(v * D[c] for c, v in w.items())
gains = []
for k, (a, b) in enumerate(SKF(5, shuffle=True, random_state=3).split(np.zeros(n), y)):
    w = climb(a); g = fast_auc(apply_w(w, O), b) - fast_auc(fixed_oof, b); gains.append(g); print(f"  held-out fold {k+1}: re-optimised weights minus fixed weights {g:+.5f}", flush=True)
gains = np.array(gains); w_full = climb(np.arange(n)); print("nested mean gain %+.5f, positive folds %d/5" % (gains.mean(), (gains > 0).sum()))
print("weights found on all rows:", {f"{t}:{nm}": round(v, 3) for (t, nm), v in sorted(w_full.items(), key=lambda kv: -kv[1])}); print("pooled held-out AUC: fixed %.5f | re-optimised %.5f (in-sample for the weights; use the nested number above)" % (roc_auc_score(y, fixed_oof), roc_auc_score(y, apply_w(w_full, O))))
PASS7 = bool(gains.mean() >= 0.0002 and (gains > 0).sum() >= 4); print("RULE:", "PASS - v13_A is worth a submission" if PASS7 else "fail - keep the fixed weights")
if not SMOKE:
    o7 = pd.DataFrame({"SK_ID_CURR": sample.SK_ID_CURR, "TARGET": apply_w(w_full, T)}); assert len(o7) == 61500 and o7.SK_ID_CURR.is_unique and np.isfinite(o7.TARGET).all(); o7.to_csv(os.path.join(OUT_DIR, "v13_A_reweighted.csv"), index=False); print("wrote v13_A_reweighted.csv (submit only if the rule passed)")


In [ ]:
# cell 8 - extra_trees versions of the five e1 members (about 1.7 hours), then the three-way blend
SPEC1_ET = {nm: (X, Xt, {**p, "extra_trees": True}, R, Rf, seeds) for nm, (X, Xt, p, R, Rf, seeds) in SPEC1.items()}
for nm in W: RESV[("e1et", nm)] = run_member(nm, SPEC1_ET, "e1et", VIEW1)
print("\n== extra_trees effect at production settings, member by member (same folds, same rounds) ==")
for nm in W: print(f"  {nm}: e1 {roc_auc_score(y, RESV[('e1', nm)][1]):.5f} -> e1 + extra_trees {roc_auc_score(y, RESV[('e1et', nm)][1]):.5f}  ({roc_auc_score(y, RESV[('e1et', nm)][1]) - roc_auc_score(y, RESV[('e1', nm)][1]):+.5f})")
print(f"  blend of the five: e1 {roc_auc_score(y, blend('e1', 0)):.5f} -> e1et {roc_auc_score(y, blend('e1et', 0)):.5f} | rank corr of the two blends {pd.Series(blend('e1', 0)).corr(pd.Series(blend('e1et', 0)), method='spearman'):.4f}")
folds8 = list(SKF(5, shuffle=True, random_state=42).split(np.zeros(n), y)); pos8, neg8 = np.where(y == 1)[0], np.where(y == 0)[0]; rs8 = np.random.RandomState(0)
dr8 = [np.r_[rs8.choice(pos8, len(pos8)), rs8.choice(neg8, len(neg8))] for _ in range(200)]
CANDS = {"B: old + e1 + e1et (1/3 each)": ((blend("old", 0) + blend("e1", 0) + blend("e1et", 0)) / 3, (blend("old", 1) + blend("e1", 1) + blend("e1et", 1)) / 3, "v13_B_old_e1_e1et.csv"),
         "C: old + e1et (swap e1 for e1et)": (0.5 * blend("old", 0) + 0.5 * blend("e1et", 0), 0.5 * blend("old", 1) + 0.5 * blend("e1et", 1), "v13_C_old_e1et_swap.csv")}
print(f"\nbaseline (submitted recipe) pooled held-out AUC {roc_auc_score(y, fixed_oof):.5f}")
for lab, (co, ct, fn) in CANDS.items():
    fg = np.array([fast_auc(co, b) - fast_auc(fixed_oof, b) for _, b in folds8]); bd = np.array([fast_auc(co, i) - fast_auc(fixed_oof, i) for i in dr8]); lo, hi = np.percentile(bd, [2.5, 97.5]); ok = fg.mean() >= 0.0002 and (fg > 0).sum() >= 4
    print(f"== {lab}: pooled {roc_auc_score(y, co):.5f} ({roc_auc_score(y, co) - roc_auc_score(y, fixed_oof):+.5f}) | per-fold {np.round(fg, 5)} | mean {fg.mean():+.5f}, positive {(fg > 0).sum()}/5 | bootstrap 95% CI (information only) [{lo:+.5f}, {hi:+.5f}] -> {'PASS' if ok else 'fail'}")
    if not SMOKE:
        o8 = pd.DataFrame({"SK_ID_CURR": sample.SK_ID_CURR, "TARGET": ct}); assert len(o8) == 61500 and o8.SK_ID_CURR.is_unique and np.isfinite(o8.TARGET).all(); o8.to_csv(os.path.join(OUT_DIR, fn), index=False); print("   wrote", fn, "(submit only if PASS)")
